In [57]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [58]:
df =pd.read_csv("dataset.csv")

In [59]:
df.head(5)

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [60]:
df = df.drop([df.columns[0], "track_id"], axis=1)


In [61]:
df.head(5)

,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [62]:
print(df.isnull().sum())

artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64


In [63]:
df['artists'] = df['artists'].fillna('Unknown')
df['album_name'] = df['album_name'].fillna('Unknown')
df['track_name'] = df['track_name'].fillna('Unknown')



In [64]:

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['track_genre'] = le.fit_transform(df['track_genre'])

In [65]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
columns_to_normalize = ['danceability', 'energy', 'key', 
                        'loudness', 'mode', 'speechiness', 
                        'acousticness', 'instrumentalness', 
                        'liveness', 'valence', 'tempo','popularity','duration_ms','time_signature','track_genre']


In [66]:

df_normalized = scaler.fit_transform(df[columns_to_normalize])

# Normalleştirilmiş veriyi DataFrame'e çevirip, orijinal kolonlara isim vermek
df_normalized = pd.DataFrame(df_normalized, columns=columns_to_normalize)

# Orijinal DataFrame'den normalleştirilmiş kolonları çıkarıyoruz
df_original = df.drop(columns=columns_to_normalize)


df_final = pd.concat([df_original, df_normalized], axis=1)
pd.set_option('display.max_columns', None)  
pd.set_option('display.max_rows', None) 
pd.set_option('display.width', None)  
pd.set_option('display.max_colwidth', None)  

# Sonuç
print(df_final.head())

                  artists  \
0             Gen Hoshino   
1            Ben Woodward   
2  Ingrid Michaelson;ZAYN   
3            Kina Grannis   
4        Chord Overstreet   

                                               album_name  \
0                                                  Comedy   
1                                        Ghost (Acoustic)   
2                                          To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Soundtrack)   
4                                                 Hold On   

                   track_name  explicit  danceability  energy       key  \
0                      Comedy     False      0.686294  0.4610  0.090909   
1            Ghost - Acoustic     False      0.426396  0.1660  0.090909   
2              To Begin Again     False      0.444670  0.3590  0.000000   
3  Can't Help Falling In Love     False      0.270051  0.0596  0.000000   
4                     Hold On     False      0.627411  0.4430  0.181818   

   loud

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

def content_based_recommendations(input_song_name, num_recommendations=5):
    # Şarkı veri setinde mi kontrol et
    if input_song_name not in df_final['track_name'].values:
        print(f"'{input_song_name}' not found in the dataset. Please enter a valid song name.")
        return

    input_song_index = df_final[df_final['track_name'] == input_song_name].index[0]

    tfidf_vectorizer = TfidfVectorizer(stop_words='english')

    # 'Artists', 'Album Name', 'Track Name' metin sütunlarını birleştirip TF-IDF uygulama
    df_final['combined_text'] = df_final['artists'] + " " + df_final['album_name'] + " " + df_final['track_name']
    tfidf_matrix = tfidf_vectorizer.fit_transform(df_final['combined_text'])

    text_similarity = cosine_similarity(tfidf_matrix[input_song_index], tfidf_matrix).flatten()

    numeric_similarity = cosine_similarity([df_normalized.iloc[input_song_index]], df_normalized).flatten()
    
    alpha = 0.001  
    beta = 0.999  

    final_similarity = alpha * text_similarity + beta * numeric_similarity

    similar_song_indices = final_similarity.argsort()[::-1][1:num_recommendations + 1]

    recommendations = df_final.iloc[similar_song_indices][['track_name', 'artists', 'album_name']]
    return recommendations


In [68]:
recommendations = content_based_recommendations('To Begin Again', num_recommendations=20)
print(recommendations)

                                    track_name                 artists  \
8352                                 Telescope       Cage The Elephant   
71                    Superman (It's Not Easy)       Five For Fighting   
68           In My Veins - Feat. Erin Mccarley            Andrew Belle   
12082                                       紅豆               Faye Wong   
15875                        i loved you first                    joan   
15367                                 Reckless                    Lund   
11830            Play With Fire - Mono Version      The Rolling Stones   
12467                             償還 - "紅豆"廣東版               Faye Wong   
222                                Nangangamba            Zack Tabudlo   
11269        Golden Slumbers - Remastered 2009             The Beatles   
11714  Will You Still Love Me Tomorrow? - 2011           Amy Winehouse   
9838           Casa No Campo - Remastered 2021             Elis Regina   
9841                          Na Unção

In [69]:
X = []
y = []
sequence_length=5
import numpy as np 
for i in range(len(df_normalized) - sequence_length):
    X.append(df_normalized.iloc[i:i + sequence_length].values)  # Sequence of features
    y.append(df_normalized.iloc[i + sequence_length].values)    # Target is the next song's features

# Diziye dönüştürme
X, y = np.array(X), np.array(y)

# X ve y'nin şekli
print(X.shape)
print(y.shape)

(113995, 5, 15)
(113995, 15)


In [70]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Veri ön işleme (örnek)
sequence_length = 100  # Örneğin, her örneğin uzunluğu 100 zaman adımı
audio_features = ['danceability', 'energy', 'key', 'loudness', 'valence']  # Audio özellikleri

# Modeli tanımlama
model = Sequential([
    LSTM(64, activation='relu', input_shape=(sequence_length, len(columns_to_normalize)), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(15, activation='linear')  # 15 nöron, çünkü hedef verinizin boyutu 15
])


# Modeli derleme
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Model özetini yazdırma
model.summary()

c:\Users\Monster\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 100, 64)        │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,391 (130.43 KB)

 Trainable params: 33,391 (130.43 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=8,
    verbose=1
)

Epoch 1/5
11400/11400 ━━━━━━━━━━━━━━━━━━━━ 61s 5ms/step - loss: 0.0559 - mae: 0.1660 - val_loss: 0.0437 - val_mae: 0.1379
Epoch 2/5
11400/11400 ━━━━━━━━━━━━━━━━━━━━ 55s 5ms/step - loss: 0.0443 - mae: 0.1410 - val_loss: 0.0429 - val_mae: 0.1361
Epoch 3/5
 1245/11400 ━━━━━━━━━━━━━━━━━━━━ 40s 4ms/step - loss: 0.0443 - mae: 0.1406

In [ ]:
y_pred = model.predict(X_test)

713/713 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [17]:
recommendations = y_pred.argsort(axis=1)[:, -5:]  # Her örnek için en yüksek puanlı 5 öneri
catalog = list(range(y_test.shape[1]))  # Tüm hedef öğelerin listesi

In [53]:
# Metrik Fonksiyonları
def calculate_mae(test_titles, recommendations): #YENİ EKLEDİM
    errors = [1 if title not in recommendations else 0 for title in test_titles]
    return np.mean(errors)

def calculate_rmse(test_titles, recommendations): #YENİ EKLEDİM
    errors = [1 if title not in recommendations else 0 for title in test_titles]
    return np.sqrt(np.mean(np.square(errors)))

def calculate_ndcg(test_titles, recommendations):
    relevance_scores = [1 if title in test_titles else 0 for title in recommendations]
    return sum(relevance_scores) / len(recommendations)  # Basitleştirilmiş nDCG

def calculate_arhr(test_titles, recommendations):
    ranks = [1 / (rank + 1) for rank, title in enumerate(recommendations) if title in test_titles]
    return sum(ranks)

def calculate_rhr(test_titles, recommendations):
    for rank, title in enumerate(recommendations):
        if title in test_titles:
            return 1 / (rank + 1)
    return 0

def calculate_precision(test_titles, recommendations):
    relevant = [title for title in recommendations if title in test_titles]
    return len(relevant) / len(recommendations) if recommendations else 0

def calculate_recall(test_titles, recommendations):
    relevant = [title for title in recommendations if title in test_titles]
    return len(relevant) / len(test_titles) if test_titles else 0

def calculate_f1_score(precision, recall):
    return 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

def calculate_chr(test_titles, recommendations): #YENİ EKLEDİM
    hits = sum(1 for title in test_titles if title in recommendations)
    return hits / len(test_titles) if test_titles else 0

def calculate_serendipity(test_titles, recommendations, catalog): #YENİ EKLEDİM
    unexpected = [title for title in recommendations if title not in test_titles]
    return len(unexpected) / len(recommendations) if recommendations else 0

def calculate_novelty(recommendations, item_popularity):#YENİ EKLEDİM
    novelty_scores = [1 - item_popularity.get(title, 0) for title in recommendations]
    return np.mean(novelty_scores)

def calculate_personalized_novelty(recommendations, user_history):#YENİ EKLEDİM
    novelty_scores = [1 if title not in user_history else 0 for title in recommendations]
    return np.mean(novelty_scores)

def calculate_coverage(test_titles, catalog):
    recommended_set = set(catalog)
    return len(recommended_set & set(test_titles)) / len(catalog) if catalog else 0

def calculate_diversity(recommendations):
    # Örnek çeşitlilik hesaplama: Her önerinin birbirine olan farkını ölçmek
    return np.random.rand()  # Gerçek çeşitlilik hesaplaması eklenebilir

def calculate_freshness(recommendations, churn_rate):
    return 1 - churn_rate  # Daha dinamik öneriler için tazelik ölçümü

# Ana Değerlendirme Fonksiyonu
def evaluate_recommendations(test_titles, recommendations, catalog, item_popularity=None, user_history=None, churn_rate=0.2): #BURADA TANIMLAMA DEĞİŞTİĞİ İÇİN BURASI KOMPLE DEĞİŞTİ. MANTIK OLARAK BİR ŞEY DEĞİŞTİRMEDİM. MAE, RMSE VS EKLENDİĞİ İÇİN ONA GÖRE EKLEME YAPTIM.
    
    if item_popularity is None:
        item_popularity = {}
    if user_history is None:
        user_history = []

    # Hesaplanan metrikler
    precision = calculate_precision(test_titles, recommendations)
    recall = calculate_recall(test_titles, recommendations)
    f1_score = calculate_f1_score(precision, recall)
    
    # Metrik sözlüğü
    metrics = {
        "MAE": calculate_mae(test_titles, recommendations),
        "RMSE": calculate_rmse(test_titles, recommendations),
        "nDCG": calculate_ndcg(test_titles, recommendations),
        "ARHR": calculate_arhr(test_titles, recommendations),
        "RHR": calculate_rhr(test_titles, recommendations),
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1_score,
        "CHR": calculate_chr(test_titles, recommendations),
        "Serendipity": calculate_serendipity(test_titles, recommendations, catalog),
        "Novelty": calculate_novelty(recommendations, item_popularity),
        "Personalized Novelty": calculate_personalized_novelty(recommendations, user_history),
        "Diversity": calculate_diversity(recommendations),
        "Coverage": calculate_coverage(test_titles, catalog),
        "Freshness": calculate_freshness(recommendations, churn_rate),
    }
    
    return metrics



In [54]:
if __name__ == "__main__": #SENİN SON 2 CODE BLOCKUNUN YERİNE BURAYI KOYDUM.
    # Test veri seti
    test_titles = ["item1", "item2", "item3"]
    recommendations = ["item2", "item4", "item3", "item5"]
    catalog = ["item1", "item2", "item3", "item4", "item5", "item6"]
    item_popularity = {"item1": 0.8, "item2": 0.6, "item3": 0.7, "item4": 0.3, "item5": 0.2, "item6": 0.1}
    user_history = ["item1", "item6"]

    # Değerlendirme
    metrics = evaluate_recommendations(test_titles, recommendations, catalog, item_popularity, user_history)

    # Sonuçları Yazdır
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")


MAE: 0.3333
RMSE: 0.5774
nDCG: 0.5000
ARHR: 1.3333
RHR: 1.0000
Precision: 0.5000
Recall: 0.6667
F1-Score: 0.5714
CHR: 0.6667
Serendipity: 0.5000
Novelty: 0.5500
Personalized Novelty: 1.0000
Diversity: 0.1428
Coverage: 0.5000
Freshness: 0.8000


In [56]:
if 'track_name' in df.columns:
    random_10_tracks = df['track_name'].sample(n=10)  # Random 10 şarkı seçimi
    print("Datasetteki Rastgele 10 Şarkı Adı:")
    for idx, track in enumerate(random_10_tracks, start=1):
        print(f"{idx}: {track}")
else:
    print("Dataset içinde 'track_name' sütunu bulunamadı.")



Datasetteki İlk 10 Şarkı Adı:
1: Comedy
2: Ghost - Acoustic
3: To Begin Again
4: Can't Help Falling In Love
5: Hold On
6: Days I Will Remember
7: Say Something
8: I'm Yours
9: Lucky
10: Hunger


In [55]:
# Kullanıcıdan şarkı adı al
user_input = input("Lütfen bir şarkı adı girin: ").strip()

# Test şarkısını bul
test_song = df[df['track_name'].str.contains(user_input, case=False)]
if test_song.empty:
    print("Girilen şarkı adı veri kümesinde bulunamadı.")
else:
    print(f"Test için seçilen şarkı: {test_song['track_name'].values[0]}")

    # Test şarkısının bilgileri
    test_titles = test_song['track_name'].tolist()
    catalog = df['track_name'].tolist()
    # Item popularity için sözlük oluşturma
    item_popularity = {title: idx / len(catalog) if len(catalog) > 0 else 0 for idx, title in enumerate(catalog)}

    user_history = []  # Varsayılan olarak boş bırakılmıştır

    # Modelin önerileri (örnek veri - modeli entegre edin)
    recommendations = ["Recommendation 1", "Recommendation 2", test_titles[0]]  # Örnek öneriler

    # Metriklerin hesaplanması
    metrics = evaluate_recommendations(
        test_titles=test_titles,
        recommendations=recommendations,
        catalog=catalog,
        churn_rate=0.2,
    )

    # Metriklerin yazdırılması
    print("Hesaplanan Metrikler:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")


Test için seçilen şarkı: Comedy
Hesaplanan Metrikler:
MAE: 0.2000
RMSE: 0.4472
nDCG: 0.3333
ARHR: 0.3333
RHR: 0.3333
Precision: 0.3333
Recall: 0.2000
F1-Score: 0.2500
CHR: 0.8000
Serendipity: 0.6667
Novelty: 1.0000
Personalized Novelty: 1.0000
Diversity: 0.1995
Coverage: 0.0000
Freshness: 0.8000
